In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("../Data/UCI_Credit_Card.csv")
df = pd.get_dummies(df, columns=["SEX"], dtype=int)
df = pd.get_dummies(df, columns=["EDUCATION"], dtype=int)
df = pd.get_dummies(df, columns=["MARRIAGE"], dtype=int)
pay_columns = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
df = pd.get_dummies(df, columns=pay_columns, dtype=int)

x = df.drop(columns=["ID", "default.payment.next.month"])
y = df["default.payment.next.month"]

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# 1. 切分資料：80% 訓練集、20% 驗證集
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=18,
    shuffle=True,
    stratify=y  # 讓兩組資料的違約／未違約比例接近原始資料
)

# 2. 建立並訓練決策樹，只使用訓練資料
model = DecisionTreeClassifier(random_state=6)
model.fit(x_train, y_train)

# 3. 取得驗證資料的預測違約機率
# predict_proba 的每一欄對應 model.classes_ 中的一個類別
positive_index = np.where(model.classes_ == 1)[0][0]
y_prob = model.predict_proba(x_val)[:, positive_index]

# 整理真實標籤與預測機率，索引對應原始資料列
prediction_scores = pd.DataFrame({
    "真實標籤": y_val,
    "預測違約機率": y_prob
})

print("Validation Dataset 前 10 筆預測結果：")
print(prediction_scores.head(10))

# 4. 使用連續機率計算 ROC 與 AUC
fpr, tpr, thresholds = roc_curve(y_val, y_prob, pos_label=1)
auc_value = roc_auc_score(y_val, y_prob)

print(f"\nValidation AUC：{auc_value:.4f}")

# 5. 繪製 ROC Curve，並將 AUC 標示在圖例
fig = plt.figure(figsize=(9, 5))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"(AUC = {auc_value:.4f})"
)
plt.plot(
    [0, 1], [0, 1],
    "k--",
    label="Random Classifier (AUC = 0.5)"
)

plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve - Validation Dataset")
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("../Data/decision_tree_ROC_curve.png", dpi=300, bbox_inches="tight")
plt.close(fig)

Validation Dataset 前 10 筆預測結果：
       真實標籤  預測違約機率
17936     0     0.0
15852     0     0.0
11601     1     0.0
664       0     1.0
13345     0     0.0
25150     0     0.0
21406     1     0.0
16678     0     0.0
15095     1     0.0
18160     0     0.0

Validation AUC：0.6299
